In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import gc
import torch

# Delete any old model variables if they exist in the namespace
if 'model' in locals():
    del model
if 'optimizer' in locals():
    del optimizer

# Force garbage collection and empty the PyTorch cache
gc.collect()
torch.cuda.empty_cache()

In [3]:
"""
  Architecture:
    - TF-IDF feature extraction (manual, from scratch)
    - 3-layer MLP with:
        • ReLU activations
        • Dropout regularisation
        • Batch Normalisation
    - Softmax output over 5 options
    - Cross-entropy loss with label smoothing
    - Adam optimiser (implemented from scratch)
    - 5-fold cross validation
    - W&B logging
"""

import os, math, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import wandb
warnings.filterwarnings("ignore")
os.environ["WANDB_SILENT"] = "true"

### CONFIG

In [4]:
DATA_DIR    = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
OPTION_COLS = ["A", "B", "C", "D", "E"]
N_FOLDS     = 5
SEED        = 42
np.random.seed(SEED)

# MLP hyperparams
HIDDEN_DIMS    = [512, 256, 128]
DROPOUT_RATE   = 0.3
LEARNING_RATE  = 1e-3
EPOCHS         = 10
BATCH_SIZE     = 64
LABEL_SMOOTH   = 0.1
MAX_FEATURES   = 20_000   

### W&B

In [5]:
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

wandb.login(key=WANDB_API_KEY)
wandb.init(
    project="24f2000817-t22026",
    name="mlp-scratch",
    config=dict(
        hidden_dims=HIDDEN_DIMS, dropout=DROPOUT_RATE,
        lr=LEARNING_RATE, epochs=EPOCHS, batch_size=BATCH_SIZE,
        label_smooth=LABEL_SMOOTH, max_features=MAX_FEATURES,
        n_folds=N_FOLDS,
    ),
)

### MAP@3

In [6]:
def apk(actual, predicted, k=3):
    if not actual: return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])

### DATA

In [7]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
train_df = train_df.dropna(subset=["prompt"] + OPTION_COLS + ["answer"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["prompt"] + OPTION_COLS).reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

label_map = {c: i for i, c in enumerate(OPTION_COLS)}
train_df["label_idx"] = train_df["answer"].map(label_map)

Train: 2000  |  Test: 500


#### TF-IDF FEATURE EXTRACTION

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("\nFitting TF-IDF vectorisers (sklearn) ...")

all_prompts = train_df["prompt"].tolist() + test_df["prompt"].tolist()
all_options = []
for df_ in [train_df, test_df]:
    for _, row in df_.iterrows():
        for c in OPTION_COLS:
            all_options.append(str(row[c]))

tfidf_p = TfidfVectorizer(
    ngram_range=(1, 2), max_features=MAX_FEATURES,
    sublinear_tf=True, strip_accents="unicode",
)
tfidf_o = TfidfVectorizer(
    ngram_range=(1, 2), max_features=MAX_FEATURES // 2,
    sublinear_tf=True, strip_accents="unicode",
)
tfidf_p.fit(all_prompts)
tfidf_o.fit(all_options)

feat_dim = len(tfidf_p.vocabulary_) + len(tfidf_o.vocabulary_)
print(f"  Feature dim per option: {feat_dim}")

def make_X(df):
    """Returns X of shape (N, 5, feat_dim) as a dense float32 array."""
    N = len(df)
    prompts = df["prompt"].astype(str).tolist()
    pv_all  = tfidf_p.transform(prompts).toarray().astype(np.float32)  # (N, Vp)

    X = np.zeros((N, 5, feat_dim), dtype=np.float32)
    for j, c in enumerate(OPTION_COLS):
        opts_j = df[c].astype(str).tolist()
        ov_j   = tfidf_o.transform(opts_j).toarray().astype(np.float32)  # (N, Vo)
        X[:, j, :] = np.concatenate([pv_all, ov_j], axis=1)
    return X

print("  Transforming train ...")
X_train_5 = make_X(train_df)   # (N_train, 5, D)
y_train    = train_df["label_idx"].values

print("  Transforming test ...")
X_test_5  = make_X(test_df)    # (N_test, 5, D)



Fitting TF-IDF vectorisers (sklearn) ...
  Feature dim per option: 12971
  Transforming train ...
  Transforming test ...


### MLP FROM SCRATCH

In [9]:
def relu(x):         return np.maximum(0, x)
def relu_grad(x):    return (x > 0).astype(np.float32)

def softmax(x):
    ex = np.exp(x - x.max(axis=-1, keepdims=True))
    return ex / ex.sum(axis=-1, keepdims=True)

# ── Loss ─────────────────────────────────────────────────
def cross_entropy_loss(probs, labels, smooth=0.1, n_classes=5):
    """Label-smoothed cross-entropy."""
    N = len(labels)
    smooth_labels = np.full((N, n_classes), smooth / n_classes, dtype=np.float32)
    smooth_labels[np.arange(N), labels] += (1 - smooth)
    log_probs = np.log(probs + 1e-9)
    return -np.sum(smooth_labels * log_probs) / N, smooth_labels

# ── Batch Norm (running stats for inference) ─────────────
class BatchNorm:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.gamma   = np.ones(dim,  dtype=np.float32)
        self.beta    = np.zeros(dim, dtype=np.float32)
        self.eps     = eps
        self.mom     = momentum
        self.run_mean = np.zeros(dim, dtype=np.float32)
        self.run_var  = np.ones(dim,  dtype=np.float32)
        # cache for backward
        self.xhat = self.std = self.xmu = None

    def forward(self, x, training=True):
        if training:
            mu  = x.mean(axis=0)
            var = x.var(axis=0)
            self.run_mean = (1 - self.mom) * self.run_mean + self.mom * mu
            self.run_var  = (1 - self.mom) * self.run_var  + self.mom * var
        else:
            mu  = self.run_mean
            var = self.run_var
        self.std  = np.sqrt(var + self.eps)
        self.xmu  = x - mu
        self.xhat = self.xmu / self.std
        return self.gamma * self.xhat + self.beta

    def backward(self, dout):
        N    = dout.shape[0]
        dxhat = dout * self.gamma
        dgamma = (dout * self.xhat).sum(axis=0)
        dbeta  = dout.sum(axis=0)
        dx = (1/N) / self.std * (
            N * dxhat
            - dxhat.sum(axis=0)
            - self.xhat * (dxhat * self.xhat).sum(axis=0)
        )
        return dx, dgamma, dbeta

# ── Linear layer ─────────────────────────────────────────
class Linear:
    def __init__(self, in_dim, out_dim):
        # He initialisation
        scale = np.sqrt(2.0 / in_dim)
        self.W = (np.random.randn(in_dim, out_dim) * scale).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        self.x = None
        # Adam state
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        dW = self.x.T @ dout
        db = dout.sum(axis=0)
        dx = dout @ self.W.T
        return dx, dW, db

# ── Dropout ──────────────────────────────────────────────
class Dropout:
    def __init__(self, rate=0.3):
        self.rate = rate
        self.mask = None

    def forward(self, x, training=True):
        if training:
            self.mask = (np.random.rand(*x.shape) > self.rate).astype(np.float32)
            return x * self.mask / (1 - self.rate)
        return x

    def backward(self, dout):
        return dout * self.mask / (1 - self.rate)

# ── MLP ──────────────────────────────────────────────────
class MLP:
    """
    Input  : (B, feat_dim)   — one option's feature vector
    Output : (B, 1)          — score for that option
    Hidden : HIDDEN_DIMS layers with BN + ReLU + Dropout
    """
    def __init__(self, in_dim, hidden_dims, dropout_rate):
        self.layers  = []
        self.bns     = []
        self.drops   = []
        self.relus   = []   # cache pre-activation for grad
        prev = in_dim
        for h in hidden_dims:
            self.layers.append(Linear(prev, h))
            self.bns.append(BatchNorm(h))
            self.drops.append(Dropout(dropout_rate))
            prev = h
        self.out_layer = Linear(prev, 1)
        self.t = 0   # Adam step counter

    def forward(self, x, training=True):
        self.pre_acts = []
        h = x
        for lin, bn, drop in zip(self.layers, self.bns, self.drops):
            h = lin.forward(h)
            h = bn.forward(h, training)
            self.pre_acts.append(h.copy())
            h = relu(h)
            h = drop.forward(h, training)
        return self.out_layer.forward(h)   # (B, 1)

    def backward(self, dout):
        grads = {}
        # output layer
        dx, dW, db = self.out_layer.backward(dout)
        grads["out_W"] = dW
        grads["out_b"] = db
        # hidden layers in reverse
        for i in reversed(range(len(self.layers))):
            dx = self.drops[i].backward(dx)
            dx = dx * relu_grad(self.pre_acts[i])
            dx, dgamma, dbeta = self.bns[i].backward(dx)
            grads[f"bn_gamma_{i}"] = dgamma
            grads[f"bn_beta_{i}"]  = dbeta
            dx, dW, db = self.layers[i].backward(dx)
            grads[f"W_{i}"] = dW
            grads[f"b_{i}"] = db
        return grads

    def adam_update(self, grads, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        t = self.t

        def update(param, grad, m, v):
            m[:] = beta1 * m + (1 - beta1) * grad
            v[:] = beta2 * v + (1 - beta2) * grad**2
            mhat = m / (1 - beta1**t)
            vhat = v / (1 - beta2**t)
            param -= lr * mhat / (np.sqrt(vhat) + eps)

        update(self.out_layer.W, grads["out_W"], self.out_layer.mW, self.out_layer.vW)
        update(self.out_layer.b, grads["out_b"], self.out_layer.mb, self.out_layer.vb)

        for i, (lin, bn) in enumerate(zip(self.layers, self.bns)):
            update(lin.W, grads[f"W_{i}"], lin.mW, lin.vW)
            update(lin.b, grads[f"b_{i}"], lin.mb, lin.vb)
            bn.gamma -= lr * grads[f"bn_gamma_{i}"]
            bn.beta  -= lr * grads[f"bn_beta_{i}"]


# ── Forward pass over all 5 options → softmax scores ─────
def score_options(model, X5, training=True):
    """
    X5 shape: (N, 5, D)
    Returns logits (N, 5) by scoring each option independently.
    """
    N = X5.shape[0]
    scores = np.zeros((N, 5), dtype=np.float32)
    for j in range(5):
        scores[:, j] = model.forward(X5[:, j, :], training).squeeze(-1)
    return scores

### TRAINING LOOP (one fold)

In [10]:
def train_fold(fold, tr_idx, vl_idx,global_step):
    print(f"\n{'='*50}\nFOLD {fold+1}/{N_FOLDS}\n{'='*50}")

    X_tr, y_tr = X_train_5[tr_idx], y_train[tr_idx]
    X_vl, y_vl = X_train_5[vl_idx], y_train[vl_idx]
    N_tr = len(X_tr)

    model = MLP(feat_dim, HIDDEN_DIMS, DROPOUT_RATE)
    best_map3   = 0.0
    best_weights = None

    for epoch in range(EPOCHS):
        # shuffle
        idx = np.random.permutation(N_tr)
        X_tr, y_tr = X_tr[idx], y_tr[idx]

        epoch_loss, n_correct, n_total = 0.0, 0, 0

        for start in range(0, N_tr, BATCH_SIZE):
            xb = X_tr[start:start + BATCH_SIZE]   # (B, 5, D)
            yb = y_tr[start:start + BATCH_SIZE]   # (B,)
            B  = len(xb)

            # ── forward: score all 5 options ─────────────
            scores = score_options(model, xb, training=True)  # (B, 5)
            probs  = softmax(scores)                           # (B, 5)

            loss, smooth_labels = cross_entropy_loss(probs, yb, LABEL_SMOOTH)
            epoch_loss += loss * B
            n_correct  += (probs.argmax(axis=1) == yb).sum()
            n_total    += B

            # ── backward: dL/d(scores) ───────────────────
            dscores = (probs - smooth_labels) / B   # (B, 5)

            # backprop through each option's score independently
            total_grads = None
            for j in range(5):
                dout_j = dscores[:, j:j+1]              # (B, 1)
                # re-forward option j to rebuild cache
                _ = model.forward(xb[:, j, :], training=True)
                grads_j = model.backward(dout_j)
                if total_grads is None:
                    total_grads = grads_j
                else:
                    for k in grads_j:
                        total_grads[k] = total_grads[k] + grads_j[k]

            model.adam_update(total_grads, LEARNING_RATE)
            global_step += 1

            wandb.log({
                f"fold{fold+1}/train/step_loss": loss,
            }, step=global_step)

        epoch_loss /= n_total
        epoch_acc   = n_correct / n_total

        # ── validation ───────────────────────────────────
        val_scores = score_options(model, X_vl, training=False)
        val_preds  = [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3]
                      for r in val_scores]
        val_map3   = mapk([OPTION_COLS[l] for l in y_vl], val_preds)

        print(f"  Ep{epoch+1:02d} loss={epoch_loss:.4f}  "
              f"acc={epoch_acc:.4f}  val_MAP@3={val_map3:.4f}")

        wandb.log({
            f"fold{fold+1}/train/epoch_loss": epoch_loss,
            f"fold{fold+1}/train/epoch_acc":  epoch_acc,
            f"fold{fold+1}/val/map3":         val_map3,
            "epoch": epoch + 1,
        }, step=global_step)

        if val_map3 > best_map3:
            best_map3    = val_map3
            # deep copy weights
            best_weights = {
                "layers":    [(l.W.copy(), l.b.copy()) for l in model.layers],
                "out":       (model.out_layer.W.copy(), model.out_layer.b.copy()),
                "bn_gamma":  [bn.gamma.copy() for bn in model.bns],
                "bn_beta":   [bn.beta.copy()  for bn in model.bns],
                "bn_rmean":  [bn.run_mean.copy() for bn in model.bns],
                "bn_rvar":   [bn.run_var.copy()  for bn in model.bns],
            }
            wandb.run.summary[f"fold{fold+1}/best_val_map3"] = best_map3
            print(f"         ✓ best saved (val MAP@3={best_map3:.4f})")

    # restore best weights
    for i, (W, b) in enumerate(best_weights["layers"]):
        model.layers[i].W, model.layers[i].b = W, b
    model.out_layer.W, model.out_layer.b = best_weights["out"]
    for i, bn in enumerate(model.bns):
        bn.gamma    = best_weights["bn_gamma"][i]
        bn.beta     = best_weights["bn_beta"][i]
        bn.run_mean = best_weights["bn_rmean"][i]
        bn.run_var  = best_weights["bn_rvar"][i]

    # test inference
    test_scores = score_options(model, X_test_5, training=False)
    return test_scores, best_map3, global_step

### 5-FOLD CV

In [11]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_test_scores = []
fold_scores     = []
global_step     = 0   

for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df, y_train)):
    scores, best, global_step = train_fold(fold, tr_idx, vl_idx,global_step)
    all_test_scores.append(scores)
    fold_scores.append(best)

print(f"\nCV MAP@3 per fold : {[f'{s:.4f}' for s in fold_scores]}")
print(f"Mean CV MAP@3     : {np.mean(fold_scores):.4f}")

wandb.run.summary["cv_map3_mean"] = float(np.mean(fold_scores))
wandb.run.summary["cv_map3_std"]  = float(np.std(fold_scores))
wandb.log({
    "cv_fold_map3": wandb.plot.bar(
        wandb.Table(
            columns=["fold", "val_map3"],
            data=[[f"fold{i+1}", s] for i, s in enumerate(fold_scores)],
        ),
        "fold", "val_map3", title="Val MAP@3 per Fold",
    )
})


FOLD 1/5
  Ep01 loss=1.3345  acc=0.5444  val_MAP@3=0.9779
         ✓ best saved (val MAP@3=0.9779)
  Ep02 loss=0.8518  acc=0.8306  val_MAP@3=0.9804
         ✓ best saved (val MAP@3=0.9804)
  Ep03 loss=0.8315  acc=0.8750  val_MAP@3=0.9804
  Ep04 loss=0.8576  acc=0.8838  val_MAP@3=0.9804
  Ep05 loss=0.8921  acc=0.8925  val_MAP@3=0.9792
  Ep06 loss=0.9701  acc=0.8881  val_MAP@3=0.9804
  Ep07 loss=1.0182  acc=0.8812  val_MAP@3=0.9754
  Ep08 loss=1.0504  acc=0.8844  val_MAP@3=0.9804
  Ep09 loss=1.1335  acc=0.8706  val_MAP@3=0.9804
  Ep10 loss=1.1551  acc=0.8944  val_MAP@3=0.9804

FOLD 2/5
  Ep01 loss=1.2508  acc=0.5487  val_MAP@3=0.9879
         ✓ best saved (val MAP@3=0.9879)
  Ep02 loss=0.8081  acc=0.8300  val_MAP@3=0.9892
         ✓ best saved (val MAP@3=0.9892)
  Ep03 loss=0.7402  acc=0.8825  val_MAP@3=0.9892
  Ep04 loss=0.7900  acc=0.8900  val_MAP@3=0.9879
  Ep05 loss=0.7991  acc=0.8925  val_MAP@3=0.9892
  Ep06 loss=0.8758  acc=0.8844  val_MAP@3=0.9892
  Ep07 loss=0.9135  acc=0.9012  

### ENSEMBLE & SUBMISSION

In [12]:
avg_scores = np.mean(all_test_scores, axis=0)   # (N_test, 5)
test_preds = [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3]
              for r in avg_scores]

submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
out_path = "/kaggle/working/submission_scratch_dl.csv"
submission.to_csv(out_path, index=False)
print(f"\n✅  Saved -> {out_path}")
print(submission.head(10).to_string(index=False))

wandb.finish()


✅  Saved -> /kaggle/working/submission_scratch_dl.csv
 ID Prediction
  1      A D B
  2      B E C
  3      B E D
  4      E C A
  5      C A D
  6      D E A
  7      E D A
  8      B E A
  9      C D E
 10      B E D
